In [1]:
# Librerias
import re
import nltk

import pandas as pd
import numpy as np
import lightgbm as lgb

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from nltk.corpus import stopwords

In [2]:
# Cargar datos
train = pd.read_csv('data/train.csv')
eval_df = pd.read_csv('data/eval.csv')

data = train.copy()

print("TAMAÑO DEL DATASET")
print(data.shape)
print(data.head())

print("\nDISTRIBUCIÓN DE LAS DECADAS")
print(data['decade'].value_counts())
print("\nNúmero de decadas:", data['decade'].nunique())

TAMAÑO DEL DATASET
(31403, 2)
                                                text  decade
0  \nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...     164
1  gone. Sus amigos , sus clientes, todo \ncuanto...     182
2  Prefosen quemanera,e per qualesfolpechas deuan...     157
3  Caistro  el  M  a  y  o  r  a  i  .]  Del  ape...     163
4  \nlos  que  panden  macho  ;  y \notros  en  l...     166

DISTRIBUCIÓN DE LAS DECADAS
decade
160    848
172    842
155    836
170    833
167    831
178    831
154    830
157    827
163    827
180    825
168    822
175    817
171    816
165    814
151    812
188    809
179    809
182    808
162    808
174    807
164    804
185    803
184    802
173    802
159    802
181    795
183    794
156    792
161    787
187    787
150    786
152    785
177    782
166    779
158    778
153    775
186    773
169    771
176    754
Name: count, dtype: int64

Número de decadas: 39


In [ ]:
# Varianza en el tamaño de los textos.
data['text_len'] = data['text'].apply(len)
print(data['text_len'].describe())

In [ ]:
# Ejemplos de distintas décadas
for decade in [150, 165, 180]:
    ejemplo = data[data['decade'] == decade]['text'].iloc[0]
    print(f"\n=== Década {decade} ===")
    print(ejemplo[:300])
    print("---")

In [3]:
# Stopwords en español - nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    # 1. Normalizar saltos de línea y espacios múltiples
    texto = re.sub(r'\n+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    
    # 2. Quitar caracteres que claramente son ruido OCR
    # (símbolos que no son letras, números ni puntuación básica)
    texto = re.sub(r'[^\w\s.,;:!?áéíóúüñÁÉÍÓÚÜÑ]', ' ', texto)
    
    # 3. Strip
    texto = texto.strip().lower()

    # 4. Quitar stopwords
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words]

    # 5. Quitar puntuación y números que quedaron sueltos
    texto = re.sub(r'\b[\d.,;:!?]+\b', ' ', texto)
    
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words]
    
    # 6. Filtrar tokens de 1 o 2 caracteres (ruido OCR)
    palabras = [p for p in palabras if len(p) > 2]

    return ' '.join(palabras)

data['text_clean'] = data['text'].apply(limpiar_texto)
eval_df['text_clean'] = eval_df['text'].apply(limpiar_texto)

# Verifica el resultado
for decade in [150, 165, 180]:
    ejemplo = data[data['decade'] == decade]['text_clean'].iloc[0]
    print(f"\n=== Década {decade} ===")
    print(ejemplo[:300])

[nltk_data] Downloading package stopwords to C:\Users\Juan
[nltk_data]     David\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



=== Década 150 ===
efiotnl fiit trce lleene. leee ittc iieii iié oii ette íléiw temer tnner ellmpí

=== Década 165 ===
efto viña confejo portagal qual pbí precedido aragón nunca queri concurrir éfl asproce fsione lámanos jun

=== Década 180 ===
obligación ordenanzas imponen director general armada zelar mejoren cartas derroteros conformidad noticias deben dársele quanto descubrimientos nuevas tierras, islas, baxos sondas, rectificación acaso hiciese posiciones locadas. cuidado duda alguna bueno pro vechoso: capaz producir armada real utili


In [ ]:
for decade in [150, 165, 180]:
    textos = ' '.join(data[data['decade'] == decade]['text_clean'])
    palabras = textos.split()
    mas_comunes = Counter(palabras).most_common(15)
    print(f"\n=== Década {decade} ===")
    print(mas_comunes)

In [4]:
data = data[data['text_clean'].str.strip() != '']

print(data[['text', 'text_clean', 'decade']].head(3))
print(f"\nTrain: {data.shape}")

                                                text  \
0  \nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...   
1  gone. Sus amigos , sus clientes, todo \ncuanto...   
2  Prefosen quemanera,e per qualesfolpechas deuan...   

                                          text_clean  decade  
0  honorarias jubiladas. dit pontem poreft proreg...     164  
1  gone. amigos clientes, cuanto rodea prueban ho...     182  
2  prefosen quemanera per qualesfolpechas deuan f...     157  

Train: (31394, 3)


In [5]:
tfidf = TfidfVectorizer(
    max_features=200000,  # más vocabulario
    ngram_range=(1, 3),   # agrega trigramas
    min_df=2,
    sublinear_tf=True,    # aplica log a la frecuencia, reduce dominancia de palabras muy frecuentes
)

X_train = tfidf.fit_transform(data['text_clean'])
y_train = data['decade']

print("\nTAMAÑO DEL DATASET")
print(X_train.shape)


TAMAÑO DEL DATASET
(31394, 136790)


In [ ]:
lr = LogisticRegression(
    max_iter=1000,  # suficientes iteraciones para que converja
    C=1.0,          # regularización por defecto
)

# Evaluamos con cross-validation (5 folds)
scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='accuracy')

print(f"Accuracy por fold: {scores}")
print(f"Accuracy promedio: {scores.mean():.4f}")
print(f"Desviación estándar: {scores.std():.4f}")

In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    C=5.0,  # menos regularización
    solver='saga',  # más eficiente para datasets grandes con muchas clases
)

scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='accuracy')

print(f"Accuracy por fold: {scores}")
print(f"Accuracy promedio: {scores.mean():.4f}")
print(f"Desviación estándar: {scores.std():.4f}")

In [ ]:
svm = LinearSVC(
    C=1.0,
    max_iter=2000,
)

scores = cross_val_score(svm, X_train, y_train, cv=5, scoring='accuracy')

print(f"Accuracy por fold: {scores}")
print(f"Accuracy promedio: {scores.mean():.4f}")
print(f"Desviación estándar: {scores.std():.4f}")

In [ ]:
for alpha in [0.01, 0.05, 0.1, 0.5, 1.0]:
    nb = MultinomialNB(alpha=alpha)
    scores = cross_val_score(nb, X_train, y_train, cv=5, scoring='accuracy')
    print(f"---- ALPHA {alpha} ----")

    print(f"Accuracy por fold: {scores}")
    print(f"Accuracy promedio: {scores.mean():.4f}")
    print(f"Desviación estándar: {scores.std():.4f}\n")

In [ ]:
lr = LogisticRegression(max_iter=1000, C=1.0)

# Obtener predicciones por cross-validation
y_pred = cross_val_predict(lr, X_train, y_train, cv=5)

# Ver cuántas predicciones están "cerca" de la real
diferencia = np.abs(y_pred - y_train)

print(f"Exactas (diff=0):     {(diferencia == 0).mean():.4f}")
print(f"Con 1 década (diff≤1): {(diferencia <= 1).mean():.4f}")
print(f"Con 2 décadas (diff≤2): {(diferencia <= 2).mean():.4f}")
print(f"Con 5 décadas (diff≤5): {(diferencia <= 5).mean():.4f}")

In [ ]:
for max_features in [300000, 500000]:
    for ngram in [(1,3), (1,4)]:
        tfidf_test = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram,
            min_df=2,
            sublinear_tf=True,
        )
        X_test = tfidf_test.fit_transform(data['text_clean'])
        nb = MultinomialNB(alpha=0.1)
        scores = cross_val_score(nb, X_test, y_train, cv=5, scoring='accuracy')
        print(f"max_features={max_features}, ngram={ngram}: {scores.mean():.4f}")

In [ ]:
# Entrenar ambos modelos
lr = LogisticRegression(max_iter=1000, C=5.0, solver='saga')
nb = MultinomialNB(alpha=0.1)

# Cross-validation manual para combinar probabilidades
skf = StratifiedKFold(n_splits=5)
accuracies = []

for train_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    lr.fit(X_tr, y_tr)
    nb.fit(X_tr, y_tr)
    
    # Promediar probabilidades
    proba_ensemble = (0.4 * lr.predict_proba(X_val) + 0.6 * nb.predict_proba(X_val))
    y_pred = lr.classes_[np.argmax(proba_ensemble, axis=1)]
    
    accuracies.append((y_pred == y_val).mean())

print(f"Accuracy ensemble: {np.mean(accuracies):.4f}")

In [ ]:
# Entrenar con todos los datos
lr.fit(X_train, y_train)
nb.fit(X_train, y_train)

# Predecir sobre eval
X_eval_final = tfidf.transform(eval_df['text_clean'])

proba_ensemble = (0.4 * lr.predict_proba(X_eval_final) + 0.6 * nb.predict_proba(X_eval_final))
y_pred_eval = lr.classes_[np.argmax(proba_ensemble, axis=1)]

submission = pd.DataFrame({
    'id': eval_df['id'],
    'answer': y_pred_eval
})

submission.to_csv('submission_ensemble.csv', index=False)
print(submission.head(10))
print(f"Shape: {submission.shape}")

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    objective='multiclass',
    num_class=39,
    n_estimators=200,
    learning_rate=0.1,
    num_leaves=63,
    n_jobs=-1,        # usa todos los núcleos disponibles
    random_state=42,
    verbosity=-1      # silencia los logs de entrenamiento
)

scores = cross_val_score(lgbm, X_train, y_train, cv=5, scoring='accuracy')

print(f"Accuracy por fold: {scores}")
print(f"Accuracy promedio: {scores.mean():.4f}")
print(f"Desviación estándar: {scores.std():.4f}")

In [6]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

# Empezamos con 200 componentes
svd = TruncatedSVD(n_components=200, random_state=42)
normalizer = Normalizer(copy=False)  # importante después de SVD

# Pipeline SVD + normalización
lsa = make_pipeline(svd, normalizer)

X_train_svd = lsa.fit_transform(X_train)

print(f"Shape original: {X_train.shape}")
print(f"Shape reducido: {X_train_svd.shape}")

Shape original: (31394, 136790)
Shape reducido: (31394, 200)


In [8]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    objective='multiclass',
    num_class=39,
    n_estimators=300,
    learning_rate=0.1,
    num_leaves=63,
    n_jobs=-1,
    random_state=42,
    verbosity=-1
)

scores = cross_val_score(lgbm, X_train_svd, y_train, cv=5, scoring='accuracy')

print(f"Accuracy por fold: {scores}")
print(f"Accuracy promedio: {scores.mean():.4f}")
print(f"Desviación estándar: {scores.std():.4f}")

c:\Users\Juan David\Downloads\MaterialDeClase-ISIS-2611\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Juan David\Downloads\MaterialDeClase-ISIS-2611\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Juan David\Downloads\MaterialDeClase-ISIS-2611\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Juan David\Downloads\MaterialDeClase-ISIS-2611\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Juan David\Downloads\MaterialDeClase-ISIS-2611\.venv\Lib\site-packages\

Accuracy por fold: [0.15034241 0.14843128 0.14699793 0.14715719 0.15402995]
Accuracy promedio: 0.1494
Desviación estándar: 0.0026


In [9]:
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=200000,
    min_df=2,
    sublinear_tf=True,
)

X_train_char = tfidf_char.fit_transform(data['text_clean'])

# Primero evalúa solo NB para ver si char n-gramas ayudan
nb = MultinomialNB(alpha=0.1)
scores = cross_val_score(nb, X_train_char, y_train, cv=5, scoring='accuracy')

print(f"Accuracy promedio: {scores.mean():.4f}")
print(f"Desviación estándar: {scores.std():.4f}")

Accuracy promedio: 0.2321
Desviación estándar: 0.0034


In [10]:
lr = LogisticRegression(max_iter=1000, C=5.0, solver='saga')
scores = cross_val_score(lr, X_train_char, y_train, cv=5, scoring='accuracy')

print(f"Accuracy promedio: {scores.mean():.4f}")
print(f"Desviación estándar: {scores.std():.4f}")

Accuracy promedio: 0.2429
Desviación estándar: 0.0038


In [11]:
skf = StratifiedKFold(n_splits=5)
accuracies = []

for train_idx, val_idx in skf.split(X_train_char, y_train):
    X_tr, X_val = X_train_char[train_idx], X_train_char[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    lr.fit(X_tr, y_tr)
    nb.fit(X_tr, y_tr)
    
    proba_ensemble = (0.4 * lr.predict_proba(X_val) + 0.6 * nb.predict_proba(X_val))
    y_pred = lr.classes_[np.argmax(proba_ensemble, axis=1)]
    
    accuracies.append((y_pred == y_val).mean())

print(f"Accuracy ensemble char: {np.mean(accuracies):.4f}")

Accuracy ensemble char: 0.2395


In [12]:
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer

def identity(x):
    return x

# Vectorizador de palabras (nuestro mejor)
tfidf_word = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    max_features=200000,
    min_df=2,
    sublinear_tf=True,
)

# Vectorizador de caracteres (nuevo)
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=200000,
    min_df=2,
    sublinear_tf=True,
)

# Combinar ambas matrices horizontalmente
X_word = tfidf_word.fit_transform(data['text_clean'])
X_char = tfidf_char.fit_transform(data['text_clean'])

from scipy.sparse import hstack
X_combined = hstack([X_word, X_char])

print(f"Shape word: {X_word.shape}")
print(f"Shape char: {X_char.shape}")
print(f"Shape combinado: {X_combined.shape}")

# Evaluar LR sobre la combinación
lr = LogisticRegression(max_iter=1000, C=5.0, solver='saga')
scores = cross_val_score(lr, X_combined, y_train, cv=5, scoring='accuracy')

print(f"\nAccuracy promedio: {scores.mean():.4f}")
print(f"Desviación estándar: {scores.std():.4f}")

Shape word: (31394, 136790)
Shape char: (31394, 200000)
Shape combinado: (31394, 336790)

Accuracy promedio: 0.2476
Desviación estándar: 0.0034
